In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
conn = sqlite3.connect("../db.sqlite3")
cursor = conn.cursor()

query = (
    "SELECT "
        "restaurant_orderitem.id as id, "
        "datetime, "
        "order_id, "
        "product_id, "
        "quantity, "
        "price, "
        "restaurant_product.name as product_name "
    "FROM restaurant_orderitem "
    "JOIN restaurant_order on restaurant_order.id = restaurant_orderitem.order_id "
    "JOIN restaurant_product on restaurant_product.id = restaurant_orderitem.product_id "
)

df = pd.read_sql_query(query, con=conn)

df.to_csv("orders.csv", index=False)

conn.close()

In [ ]:
df = pd.read_csv("orders.csv", index_col="id")
df.head()

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
top_ten_prods = (
    df[["quantity", "product_name"]]
    .groupby("product_name")
    .sum()
    .sort_values("quantity", ascending=False)
    .head(10)
)
data = top_ten_prods["quantity"]
total = data.sum()
def format_label(pct):
    raw_val = int(round(pct * total / 100))
    return f"{pct:.1f}% ({raw_val})"

plt.figure(figsize=(10, 8))

data.plot(
    kind="pie",
    autopct=format_label,
    startangle=90,
    ylabel="Quantity",
    title="Top 10 products by quantity",
)

plt.show()

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
df["item_price"] = df["price"] * df["quantity"]
top_ten_prods = (
    df[["item_price", "product_name"]]
    .groupby("product_name")
    .sum()
    .sort_values("item_price", ascending=False)
    .head(10)
)
plt.figure(figsize=(10, 8))

data = top_ten_prods["item_price"]
total = data.sum()

data.plot(
    kind="pie",
    autopct=format_label,
    startangle=90,
    ylabel="Item price",
    title="Top 10 products by revenue(item_price)",
)

plt.show()

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
df['datetime'] = pd.to_datetime(df['datetime'])
df['order_hour'] = df['datetime'].dt.hour
hourly_income = df.groupby('order_hour')['item_price'].sum()
plt.figure(figsize=(12,6))
hourly_income.plot(kind='bar', color='skyblue')

plt.xlabel('Order Hour')
plt.title('Profit by Order Hour')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
df['order_day'] = df['datetime'].dt.day_name()
daily_income = df.groupby('order_day')['item_price'].sum()
plt.figure(figsize=(12,6))
daily_income.plot(kind='bar', color='skyblue')

plt.xlabel('Order Day Of The Week')
plt.title('Profit by Order Day Of The Week')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=1)

plt.show()